# RAG vs Prompt-Only — Comparative Evaluation

**Purpose:** Side-by-side experimental comparison for the Bachelor thesis.  
**Protocol:** Same questions → same Gemini model → same temperature → blind-gradable outputs.  

| | RAG System | Prompt-Only System |
|---|---|---|
| **Engine** | `rag_engine.py` | `prompt_only_engine.py` |
| **Knowledge source** | ChromaDB (corrections + textbooks) | System prompt (curriculum encoding) |
| **Embedding model** | BGE-M3 | None |
| **LLM model** | Same Gemini | Same Gemini |
| **Temperature** | 0.15 | 0.15 |
| **Max tokens** | 4096 | 4096 |
| **Output format** | 6-part structure | 6-part structure |

---

## 0. Setup & Initialization

In [ ]:
import sys
import os
import time
import json
from datetime import datetime, timezone
from pathlib import Path

print(f"Python  : {sys.version}")
print(f"CWD     : {os.getcwd()}")
print()

In [ ]:
import config
print(f"Gemini model : {config.CHAT_MODEL_ID}")
print(f"Temperature  : 0.15 (hardcoded in both engines)")
print(f"Max tokens   : 4096 (hardcoded in both engines)")

In [ ]:
# ── Initialize RAG engine ─────────────────────────────────────
from rag_engine import TunisianMathRAG

t0 = time.monotonic()
rag_engine = TunisianMathRAG()
rag_init = time.monotonic() - t0
print(f"RAG engine ready in {rag_init:.1f}s | {rag_engine.chunk_count:,} chunks")

In [ ]:
# ── Initialize Prompt-Only engine ──────────────────────────────
from prompt_only_engine import TunisianMathPromptOnly

t0 = time.monotonic()
po_engine = TunisianMathPromptOnly()
po_init = time.monotonic() - t0
print(f"Prompt-only engine ready in {po_init:.1f}s")
print(f"\nInit comparison: RAG={rag_init:.1f}s vs Prompt-Only={po_init:.1f}s")

---
## 1. Evaluation Query Bank

**10 queries** covering the main Bac Math chapters.  
Each query will be sent to BOTH systems with the SAME mode.  
Outputs are collected for blind grading by a teacher.

In [ ]:
# ── Evaluation queries ──────────────────────────────────────────
# Design: 5 correction-mode + 5 coaching-mode, across 10 chapters
# These are realistic Bac-level questions a student would ask.

EVAL_QUERIES = [
    # ── CORRECTION MODE (dry, exam-style) ──
    {
        "id": "Q01",
        "question": "Résoudre dans C l'équation z² - 4z + 8 = 0. Écrire les solutions sous forme trigonométrique.",
        "mode": "correction",
        "chapter": "Nombres complexes",
        "difficulty": "standard",
    },
    {
        "id": "Q02",
        "question": "Soit (Un) définie par U0 = 2 et Un+1 = (1/2)Un + 1. Montrer que (Un) converge et calculer sa limite.",
        "mode": "correction",
        "chapter": "Suites numériques",
        "difficulty": "standard",
    },
    {
        "id": "Q03",
        "question": "Calculer l'intégrale I = ∫₀¹ x·eˣ dx en utilisant une intégration par parties.",
        "mode": "correction",
        "chapter": "Intégration",
        "difficulty": "standard",
    },
    {
        "id": "Q04",
        "question": "Déterminer le PGCD de 1071 et 1029 par l'algorithme d'Euclide. En déduire une relation de Bézout.",
        "mode": "correction",
        "chapter": "Arithmétique",
        "difficulty": "standard",
    },
    {
        "id": "Q05",
        "question": "Résoudre l'équation différentielle y' - 3y = 6 avec la condition y(0) = 1.",
        "mode": "correction",
        "chapter": "Équations différentielles",
        "difficulty": "standard",
    },

    # ── COACHING MODE (pedagogical) ──
    {
        "id": "Q06",
        "question": "Comment étudier la continuité et la dérivabilité d'une fonction en un point ? Donne-moi la méthode avec un exemple.",
        "mode": "coaching",
        "chapter": "Fonctions numériques",
        "difficulty": "conceptuel",
    },
    {
        "id": "Q07",
        "question": "Explique-moi comment déterminer une équation cartésienne d'un plan dans l'espace à partir de 3 points.",
        "mode": "coaching",
        "chapter": "Géométrie dans l'espace",
        "difficulty": "conceptuel",
    },
    {
        "id": "Q08",
        "question": "Dans un sac il y a 4 boules rouges et 6 bleues. On tire 3 boules sans remise. Quelle est la probabilité d'avoir exactement 2 rouges ?",
        "mode": "coaching",
        "chapter": "Probabilités",
        "difficulty": "standard",
    },
    {
        "id": "Q09",
        "question": "Résoudre dans R : ln(x+3) + ln(x-1) = ln(5). Explique chaque étape.",
        "mode": "coaching",
        "chapter": "Logarithme & Exponentielle",
        "difficulty": "standard",
    },
    {
        "id": "Q10",
        "question": "Soit f(x) = x³ - 3x + 1. Faire l'étude complète de f (domaine, limites, dérivée, tableau de variation, courbe).",
        "mode": "coaching",
        "chapter": "Études de fonctions",
        "difficulty": "complet",
    },
]

print(f"Evaluation bank: {len(EVAL_QUERIES)} queries")
print(f"  Correction mode: {sum(1 for q in EVAL_QUERIES if q['mode'] == 'correction')}")
print(f"  Coaching mode  : {sum(1 for q in EVAL_QUERIES if q['mode'] == 'coaching')}")
print(f"\nChapters covered:")
for q in EVAL_QUERIES:
    print(f"  {q['id']} | {q['chapter']:<30s} | {q['mode']:<11s} | {q['difficulty']}")

---
## 2. Run Both Systems (Side-by-Side)

For each query, we run RAG first, then Prompt-Only.  
Both answers are stored for comparison.

**Rate-limit protection:** 5s pause between queries to avoid 429 quota errors.  
**Estimated API cost:** ~20 Gemini calls (10 queries × 2 systems)  
**Estimated time:** ~10-15 min (generation + delays)

In [ ]:
# ── Run comparative evaluation ─────────────────────────────────
# Rate-limit protection: pause between API calls to avoid 429 errors.
# Gemini stable models allow ~15 RPM; we do 2 calls/query + 5s gap.

INTER_QUERY_DELAY = 5   # seconds between queries (adjust if still hitting 429)
INTER_SYSTEM_DELAY = 2  # seconds between RAG and PO calls within same query

comparison_results = []

for i, q in enumerate(EVAL_QUERIES, 1):
    print(f"\n{'='*80}")
    print(f"  QUERY {q['id']} ({i}/{len(EVAL_QUERIES)}): {q['chapter']}")
    print(f"  Mode: {q['mode']} | Difficulty: {q['difficulty']}")
    print(f"  Q: {q['question']}")
    print(f"{'='*80}")

    # ── RAG System ──
    print(f"\n  [RAG] Running...")
    rag_result = rag_engine.query(q["question"], mode=q["mode"])
    print(f"  [RAG] Done: case={rag_result.retrieval_case} "
          f"conf={rag_result.confidence} "
          f"docs={len(rag_result.selected_docs)} "
          f"ret={rag_result.retrieval_time:.2f}s "
          f"gen={rag_result.generation_time:.2f}s "
          f"total={rag_result.total_time:.2f}s")
    if rag_result.error:
        print(f"  [RAG] ERROR: {rag_result.error}")

    # ── Pause between systems to avoid quota burst ──
    print(f"\n  [DELAY] Waiting {INTER_SYSTEM_DELAY}s before Prompt-Only...")
    time.sleep(INTER_SYSTEM_DELAY)

    # ── Prompt-Only System ──
    print(f"  [PROMPT-ONLY] Running...")
    po_result = po_engine.query(q["question"], mode=q["mode"])
    print(f"  [PROMPT-ONLY] Done: "
          f"conf={po_result.confidence} "
          f"gen={po_result.generation_time:.2f}s "
          f"total={po_result.total_time:.2f}s")
    if po_result.error:
        print(f"  [PROMPT-ONLY] ERROR: {po_result.error}")

    # ── Store both ──
    comparison_results.append({
        "query_id": q["id"],
        "question": q["question"],
        "mode": q["mode"],
        "chapter": q["chapter"],
        "difficulty": q["difficulty"],
        # RAG
        "rag_answer": rag_result.answer or "",
        "rag_error": rag_result.error,
        "rag_case": rag_result.retrieval_case,
        "rag_confidence": rag_result.confidence,
        "rag_n_docs": len(rag_result.selected_docs),
        "rag_best_dist": round(rag_result.selected_docs[0].distance, 4) if rag_result.selected_docs else None,
        "rag_retrieval_time": round(rag_result.retrieval_time, 3),
        "rag_generation_time": round(rag_result.generation_time, 3),
        "rag_total_time": round(rag_result.total_time, 3),
        "rag_answer_length": len(rag_result.answer or ""),
        # Prompt-Only
        "po_answer": po_result.answer or "",
        "po_error": po_result.error,
        "po_confidence": po_result.confidence,
        "po_generation_time": round(po_result.generation_time, 3),
        "po_total_time": round(po_result.total_time, 3),
        "po_answer_length": len(po_result.answer or ""),
        "po_sys_tokens": po_result.system_prompt_tokens_approx,
    })

    print(f"\n  Answer lengths: RAG={len(rag_result.answer or '')} chars | "
          f"PO={len(po_result.answer or '')} chars")

    # ── Pause between queries ──
    if i < len(EVAL_QUERIES):
        print(f"\n  [DELAY] Waiting {INTER_QUERY_DELAY}s before next query...")
        time.sleep(INTER_QUERY_DELAY)

print(f"\n\n{'='*80}")
print(f"Comparative evaluation complete: {len(comparison_results)} queries.")

---
## 3. Side-by-Side Answer Comparison

View both answers for each query. Use this to inspect quality visually  
before sending to the teacher for blind grading.

In [ ]:
# ── Display answers side by side ─────────────────────────────────
# Run this cell for the query you want to inspect.
# Change QUERY_INDEX to view different queries (0-9).

QUERY_INDEX = 0  # ← Change this to view different queries (0-9)

r = comparison_results[QUERY_INDEX]

print(f"{'='*80}")
print(f"  {r['query_id']}: {r['chapter']} ({r['mode']})")
print(f"  Q: {r['question']}")
print(f"{'='*80}")

print(f"\n{'─'*40}")
print(f"  SYSTEM A (RAG)")
print(f"  Case={r['rag_case']} | Conf={r['rag_confidence']} | "
      f"Docs={r['rag_n_docs']} | Time={r['rag_total_time']}s")
print(f"{'─'*40}")
print(r['rag_answer'] if r['rag_answer'] else f"ERROR: {r['rag_error']}")

print(f"\n\n{'─'*40}")
print(f"  SYSTEM B (PROMPT-ONLY)")
print(f"  Conf={r['po_confidence']} | Time={r['po_total_time']}s")
print(f"{'─'*40}")
print(r['po_answer'] if r['po_answer'] else f"ERROR: {r['po_error']}")

---
## 4. Quantitative Comparison Table

Metrics that can be computed automatically (before teacher grading).

In [ ]:
# ── Comparison table ─────────────────────────────────────────────

hdr = (f"{'ID':>4s}  {'Chapter':<25s}  {'Mode':<11s}  "
       f"{'RAG_T':>6s}  {'PO_T':>6s}  "
       f"{'RAG_Len':>8s}  {'PO_Len':>8s}  "
       f"{'RAG_Conf':>9s}  {'PO_Conf':>9s}  "
       f"{'RAG_OK':>7s}  {'PO_OK':>7s}")
sep = '─' * len(hdr)

print(sep)
print(hdr)
print(sep)

for r in comparison_results:
    ch = r['chapter'][:23] + '..' if len(r['chapter']) > 25 else r['chapter']
    rag_ok = 'OK' if r['rag_answer'] else 'FAIL'
    po_ok = 'OK' if r['po_answer'] else 'FAIL'
    print(f"{r['query_id']:>4s}  {ch:<25s}  {r['mode']:<11s}  "
          f"{r['rag_total_time']:6.2f}  {r['po_total_time']:6.2f}  "
          f"{r['rag_answer_length']:8d}  {r['po_answer_length']:8d}  "
          f"{r['rag_confidence']:>9s}  {r['po_confidence']:>9s}  "
          f"{rag_ok:>7s}  {po_ok:>7s}")

print(sep)

# ── Aggregate statistics ──
rag_times = [r['rag_total_time'] for r in comparison_results]
po_times = [r['po_total_time'] for r in comparison_results]
rag_lens = [r['rag_answer_length'] for r in comparison_results]
po_lens = [r['po_answer_length'] for r in comparison_results]
rag_ok_count = sum(1 for r in comparison_results if r['rag_answer'])
po_ok_count = sum(1 for r in comparison_results if r['po_answer'])

print(f"\n  AGGREGATE STATISTICS")
print(f"  {'Metric':<30s}  {'RAG':>12s}  {'Prompt-Only':>12s}")
print(f"  {'─'*56}")
print(f"  {'Success rate':<30s}  {rag_ok_count:>10d}/10  {po_ok_count:>10d}/10")
print(f"  {'Avg response time (s)':<30s}  {sum(rag_times)/len(rag_times):>12.2f}  {sum(po_times)/len(po_times):>12.2f}")
print(f"  {'Avg answer length (chars)':<30s}  {sum(rag_lens)/len(rag_lens):>12.0f}  {sum(po_lens)/len(po_lens):>12.0f}")
print(f"  {'Total API calls':<30s}  {'10':>12s}  {'10':>12s}")
print(f"  {'Embedding calls':<30s}  {'10 (BGE-M3)':>12s}  {'0':>12s}")
print(f"  {'ChromaDB queries':<30s}  {'~20-30':>12s}  {'0':>12s}")

---
## 5. Syllabus Guard Comparison

Both systems should refuse out-of-program methods.  
This tests the syllabus guard independently.

In [ ]:
GUARD_QUERIES = [
    "Utilise la règle de L'Hôpital pour calculer lim(x→0) sin(x)/x",
    "Diagonalise la matrice A = [[2,1],[0,3]] pour calculer A^n",
]

for idx, gq in enumerate(GUARD_QUERIES):
    print(f"\n{'='*80}")
    print(f"  SYLLABUS GUARD: {gq}")
    print(f"{'='*80}")

    # RAG
    rag_r = rag_engine.query(gq, mode="correction")
    print(f"\n  [RAG] (first 500 chars):")
    print(f"  {'─'*40}")
    for line in (rag_r.answer or f"ERROR: {rag_r.error}")[:500].split('\n'):
        print(f"  {line}")

    time.sleep(2)  # rate-limit protection

    # Prompt-Only
    po_r = po_engine.query(gq, mode="correction")
    print(f"\n  [PROMPT-ONLY] (first 500 chars):")
    print(f"  {'─'*40}")
    for line in (po_r.answer or f"ERROR: {po_r.error}")[:500].split('\n'):
        print(f"  {line}")
    print()

    if idx < len(GUARD_QUERIES) - 1:
        time.sleep(5)  # pause between guard queries

---
## 6. Export for Blind Grading

Generates two files:
1. **`blind_grading_sheets.json`** — Anonymized answers (System A / System B) for the teacher
2. **`comparison_full_results.json`** — Complete results with system labels for your thesis analysis

The blind grading file uses generic labels so the teacher doesn't know which answer is RAG vs Prompt-Only.

In [ ]:
import random

# ── Blind grading export ──────────────────────────────────────────
# Randomly assign RAG/PO to System A/B for each query to avoid bias.
random.seed(42)  # Reproducible

blind_sheets = []
answer_key = []  # kept secret until after grading

for r in comparison_results:
    # Randomly decide: is RAG = System A or System B?
    rag_is_a = random.choice([True, False])

    if rag_is_a:
        sys_a_answer = r['rag_answer']
        sys_b_answer = r['po_answer']
    else:
        sys_a_answer = r['po_answer']
        sys_b_answer = r['rag_answer']

    blind_sheets.append({
        "query_id": r['query_id'],
        "question": r['question'],
        "mode": r['mode'],
        "chapter": r['chapter'],
        "system_a_answer": sys_a_answer,
        "system_b_answer": sys_b_answer,
        # Grading rubric (to be filled by teacher)
        "grading": {
            "system_a": {
                "mathematical_correctness": None,   # 0-5
                "formal_rigor": None,                # 0-5
                "official_style_alignment": None,    # 0-5
                "clarity_and_structure": None,       # 0-5
                "pedagogical_quality": None,         # 0-5
                "hallucination_detected": None,      # true/false
                "overall_score": None,               # 0-20
                "comments": "",
            },
            "system_b": {
                "mathematical_correctness": None,
                "formal_rigor": None,
                "official_style_alignment": None,
                "clarity_and_structure": None,
                "pedagogical_quality": None,
                "hallucination_detected": None,
                "overall_score": None,
                "comments": "",
            },
        },
    })

    answer_key.append({
        "query_id": r['query_id'],
        "system_a_is": "RAG" if rag_is_a else "PROMPT_ONLY",
        "system_b_is": "PROMPT_ONLY" if rag_is_a else "RAG",
    })

# ── Save blind grading sheets ──
with open("blind_grading_sheets.json", "w", encoding="utf-8") as f:
    json.dump(blind_sheets, f, ensure_ascii=False, indent=2)
print(f"Blind grading sheets saved: blind_grading_sheets.json")

# ── Save answer key (DO NOT show to teacher) ──
with open("answer_key.json", "w", encoding="utf-8") as f:
    json.dump(answer_key, f, ensure_ascii=False, indent=2)
print(f"Answer key saved: answer_key.json (keep secret until after grading!)")

# ── Save full comparison results ──
full_export = {
    "metadata": {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "model": config.CHAT_MODEL_ID,
        "rag_chunks": rag_engine.chunk_count,
        "temperature": 0.15,
        "max_tokens": 4096,
    },
    "queries": comparison_results,
    "answer_key": answer_key,
}

with open("comparison_full_results.json", "w", encoding="utf-8") as f:
    json.dump(full_export, f, ensure_ascii=False, indent=2)
print(f"Full results saved: comparison_full_results.json")

print(f"\nAnswer key (for your eyes only):")
for ak in answer_key:
    print(f"  {ak['query_id']}: A={ak['system_a_is']}, B={ak['system_b_is']}")

---
## 7. Thesis-Ready Summary Report

One-screen summary for inclusion in the thesis defense.

In [ ]:
rag_successes = sum(1 for r in comparison_results if r['rag_answer'])
po_successes = sum(1 for r in comparison_results if r['po_answer'])
avg_rag_time = sum(r['rag_total_time'] for r in comparison_results) / len(comparison_results)
avg_po_time = sum(r['po_total_time'] for r in comparison_results) / len(comparison_results)
avg_rag_len = sum(r['rag_answer_length'] for r in comparison_results) / len(comparison_results)
avg_po_len = sum(r['po_answer_length'] for r in comparison_results) / len(comparison_results)

print(f"""
╔══════════════════════════════════════════════════════════════════════════╗
║         RAG vs PROMPT-ONLY — COMPARATIVE EVALUATION REPORT            ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║  Experimental Setup                                                    ║
║    LLM Model       : {config.CHAT_MODEL_ID:<50s} ║
║    Temperature      : 0.15                                             ║
║    Max tokens        : 4096                                            ║
║    Eval queries      : {len(comparison_results):<49d} ║
║    Correction mode   : {sum(1 for q in EVAL_QUERIES if q['mode']=='correction'):<49d} ║
║    Coaching mode     : {sum(1 for q in EVAL_QUERIES if q['mode']=='coaching'):<49d} ║
║                                                                        ║
║  RAG System                                                            ║
║    DB chunks         : {rag_engine.chunk_count:<49,d} ║
║    Embedding model   : {config.EMBEDDING_MODEL_NAME:<50s}║
║    Success rate      : {rag_successes}/{len(comparison_results):<48s} ║
║    Avg total time    : {avg_rag_time:<49.2f} ║
║    Avg answer length : {avg_rag_len:<49.0f} ║
║                                                                        ║
║  Prompt-Only System                                                    ║
║    System prompt     : ~{comparison_results[0]['po_sys_tokens']} tokens{' '*(44-len(str(comparison_results[0]['po_sys_tokens'])))}║
║    Success rate      : {po_successes}/{len(comparison_results):<48s} ║
║    Avg total time    : {avg_po_time:<49.2f} ║
║    Avg answer length : {avg_po_len:<49.0f} ║
║                                                                        ║
║  Key Differences                                                       ║
║    Time overhead (RAG retrieval) : {avg_rag_time - avg_po_time:+.2f}s{' '*(36-len(f'{avg_rag_time - avg_po_time:+.2f}'))}║
║    Answer length diff            : {avg_rag_len - avg_po_len:+.0f} chars{' '*(30-len(f'{avg_rag_len - avg_po_len:+.0f}'))}║
║                                                                        ║
║  Blind grading files exported:                                         ║
║    blind_grading_sheets.json  (for teacher)                            ║
║    answer_key.json            (reveal after grading)                   ║
║    comparison_full_results.json (full data for thesis)                 ║
║                                                                        ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---
## 8. Analysis: Strengths & Weaknesses

### Expected RAG Advantages:
- **Grounded in real Tunisian exam corrections** → authentic redaction style
- **Cite-able sources** → teacher can verify claims against actual documents
- **Consistent terminology** → matches official manual phrasing
- **Lower hallucination risk** on exam-specific details (years, exercise types)

### Expected Prompt-Only Advantages:
- **Faster response** (no retrieval overhead)
- **No infrastructure dependency** (no ChromaDB, no embeddings)
- **Broader parametric knowledge** → can handle unusual question formulations
- **Self-verification block** → catches more computational errors
- **Lower setup cost** (no indexing pipeline)

### Expected Prompt-Only Weaknesses:
- **No real exam grounding** → may diverge from official Tunisian style
- **Cannot cite specific corrections** → less trustworthy for students
- **Hallucination risk** on specific exam references ("Bac 2019 Exercice 3")
- **Curriculum drift** → LLM may use methods from other countries' programs
- **Token budget** → long system prompt eats into answer generation budget

### Evaluation Criteria (for teacher blind grading):
| Criterion | Weight | Description |
|---|---|---|
| Mathematical correctness | 5/20 | Are the calculations and proofs correct? |
| Formal rigor | 5/20 | Does it follow formal mathematical writing conventions? |
| Official style alignment | 3/20 | Does it match Tunisian Bac correction style? |
| Clarity & structure | 3/20 | Is it well-organized and easy to follow? |
| Pedagogical quality | 2/20 | Does it teach effectively (coaching mode)? |
| Hallucination check | 2/20 | Are there any fabricated facts or wrong theorems? |